# Experiment 17 — HPO AutoGluon with two residual layers

This experiment applies bounded hyperparameter optimization to the final two-residual-layer design. Every layer is an AutoGluon ensemble restricted to LightGBM and CatBoost and receives the active 18-feature Experiment 9 representation.

The outer validation period is May 2024 onward. It is not used for model fitting, HPO, out-of-fold residual construction, or residual-weight selection. Within the earlier history, expanding chronological folds generate predictions for the next layer. Each fold has its own earlier-month HPO split.

Final prediction = base prediction + w1 × residual-layer-1 prediction + w2 × residual-layer-2 prediction. Both weights are estimated from chronological OOF predictions only.

**Executed result:** the HPO base ensemble scored **RMSE 470.232**, **MAE 343.271**, and **R² 0.4877**. OOF weight selection chose **w1 = 0.956** and **w2 = 0.000**; applying residual layer 1 worsened validation to **474.426 RMSE**, and layer 2 was consequently ignored. HPO and residual stacking did not improve the default 18-feature ensemble, so this run is not used as the reported best result.

In [1]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
from autogluon.common import space
from autogluon.tabular import TabularPredictor
from scipy.spatial import cKDTree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [
    Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'),
    Path('/content/drive/MyDrive/KDDM-project'),
]
project_root = next(
    (path for path in candidate_roots if (path / 'exploration/metul/preprocessing.py').exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))
from preprocessing import RentalPreprocessor, load_datasets

TARGET = 'MONTHLY_RENT'
N_SPLITS = int(os.environ.get('EXP17_OOF_SPLITS', '3'))
OOF_TRIALS_PER_MODEL = int(os.environ.get('EXP17_OOF_TRIALS_PER_MODEL', '12'))
FINAL_TRIALS_PER_MODEL = int(os.environ.get('EXP17_FINAL_TRIALS_PER_MODEL', '30'))
OOF_TIME_LIMIT = int(os.environ.get('EXP17_OOF_TIME_LIMIT', '900'))
FINAL_TIME_LIMIT = int(os.environ.get('EXP17_FINAL_TIME_LIMIT', '1800'))
RUN_ID = os.environ.get('AUTOGLUON_RUN_ID', pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
MODEL_ROOT = experiment_dir / 'ag_exp_17_two_residual_hpt' / RUN_ID

COMPACT_FEATURES = [
    'TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM',
    'approval_date_num', 'approval_month', 'coe_quota_premium_mean',
    'YEAR_COMPLETED', 'MAX_FLOOR', 'mrt_dist_nearest_km', 'mall_dist_nearest_km',
]
TARGET_FEATURES = [
    'monthly_target_lag', 'town_flat_type_target_mean', 'nearest_mrt_name',
    'nearest_tourist_attraction_name', 'nearest_monument_name',
]
BEST_FEATURES = COMPACT_FEATURES + TARGET_FEATURES

print(f'Project root: {project_root}')
print(f'Model output: {MODEL_ROOT}')
print(
    f'OOF HPO: {OOF_TRIALS_PER_MODEL} trials/model, {OOF_TIME_LIMIT}s per fit; '
    f'final HPO: {FINAL_TRIALS_PER_MODEL} trials/model, {FINAL_TIME_LIMIT}s per layer.'
)

Project root: /Users/metul/Desktop/KDDM-project
Model output: /Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_17_two_residual_hpt/20261020_123932
OOF HPO: 12 trials/model, 900s per fit; final HPO: 30 trials/model, 1800s per layer.


## HPO search space

The ranges are deliberately bounded. They cover shallow through moderately complex trees and short through long boosting schedules without allowing extreme depths or leaf counts that are unlikely to generalize chronologically. Learning rate and regularization are searched logarithmically. `num_iterations` is tuned for LightGBM and `iterations` for CatBoost.

In [2]:
HPO_HYPERPARAMETERS = {
    'GBM': {
        'num_leaves': space.Int(24, 500),
        'num_iterations': space.Int(100, 1000),
        'learning_rate': space.Real(0.01, 0.12, log=True),
        'min_data_in_leaf': space.Int(20, 250),
        'feature_fraction': space.Real(0.70, 1.00),
        'bagging_fraction': space.Real(0.70, 1.00),
        'bagging_freq': 1,
        'lambda_l1': space.Real(1e-4, 20.0, log=True),
        'lambda_l2': space.Real(1e-4, 20.0, log=True),
    },
    'CAT': {
        'depth': space.Int(5, 10),
        'iterations': space.Int(100, 1000),
        'learning_rate': space.Real(0.01, 0.12, log=True),
        'l2_leaf_reg': space.Real(1.0, 30.0, log=True),
        'random_strength': space.Real(1e-3, 10.0, log=True),
        'bagging_temperature': space.Real(0.0, 1.5),
    },
}

def hpo_kwargs(trials_per_model):
    return {
        'num_trials': trials_per_model,
        'scheduler': 'local',
        'searcher': 'random',
    }

display(pd.DataFrame({
    'setting': ['OOF folds', 'OOF trials per model', 'final trials per model', 'OOF seconds per fit', 'final seconds per layer'],
    'value': [N_SPLITS, OOF_TRIALS_PER_MODEL, FINAL_TRIALS_PER_MODEL, OOF_TIME_LIMIT, FINAL_TIME_LIMIT],
}))

,setting,value
0,OOF folds,3
1,OOF trials per model,12
2,final trials per model,30
3,OOF seconds per fit,900
4,final seconds per layer,1800


## Leakage-safe feature construction

The 24-month global lag and town × flat-type mean use labelled history only. Training rows use prior months; future rows use mappings fitted on the complete permitted history. Nearest-place names are deterministic functions of fitted coordinates and external point files.

In [3]:
class RentalTargetFeatureTransformer:
    def fit(self, history):
        self.global_mean_ = float(history[TARGET].mean())
        self.group_mean_ = history.groupby(['TOWN', 'FLAT_TYPE'])[TARGET].mean()
        self.monthly_mean_ = history.groupby('period')[TARGET].mean()
        return self

    def transform(self, frame):
        result = frame[['TOWN', 'FLAT_TYPE', 'period']].copy()
        result['monthly_target_lag'] = (
            (result['period'] - 24).map(self.monthly_mean_).fillna(self.global_mean_)
        )
        keys = result.set_index(['TOWN', 'FLAT_TYPE']).index
        group_mean = pd.Series(keys.map(self.group_mean_), index=result.index)
        result['town_flat_type_target_mean'] = group_mean.fillna(self.global_mean_)
        return result[['monthly_target_lag', 'town_flat_type_target_mean']]

    def fit_transform_train(self, history):
        self.fit(history)
        grouped = (
            history.groupby(['TOWN', 'FLAT_TYPE', 'period'], as_index=False)[TARGET]
            .agg(['sum', 'count']).reset_index()
            .sort_values(['TOWN', 'FLAT_TYPE', 'period'])
        )
        grouped['prior_sum'] = grouped.groupby(['TOWN', 'FLAT_TYPE'])['sum'].cumsum() - grouped['sum']
        grouped['prior_count'] = grouped.groupby(['TOWN', 'FLAT_TYPE'])['count'].cumsum() - grouped['count']
        grouped['town_flat_type_target_mean'] = (
            grouped['prior_sum'] / grouped['prior_count'].replace(0, np.nan)
        )
        monthly = history.groupby('period')[TARGET].agg(['sum', 'count']).sort_index()
        global_prior = (
            (monthly['sum'].cumsum() - monthly['sum'])
            / (monthly['count'].cumsum() - monthly['count']).replace(0, np.nan)
        )
        result = history[['TOWN', 'FLAT_TYPE', 'period']].merge(
            grouped[['TOWN', 'FLAT_TYPE', 'period', 'town_flat_type_target_mean']],
            on=['TOWN', 'FLAT_TYPE', 'period'], how='left',
        )
        result['town_flat_type_target_mean'] = (
            result['town_flat_type_target_mean'].fillna(result['period'].map(global_prior))
        )
        result['monthly_target_lag'] = (
            (result['period'] - 24).map(self.monthly_mean_).fillna(result['period'].map(global_prior))
        )
        return result[['monthly_target_lag', 'town_flat_type_target_mean']]

def nearest_mrt_name(base, datasets):
    stations = datasets['mrt'][['NAME', 'LATITUDE', 'LONGITUDE']].dropna()
    _, index = cKDTree(
        stations[['LATITUDE', 'LONGITUDE']].to_numpy() * [111.0, 110.96]
    ).query(base[['LATITUDE', 'LONGITUDE']].to_numpy() * [111.0, 110.96])
    return pd.Series(stations['NAME'].astype(object).to_numpy()[index], name='nearest_mrt_name')

def nearest_geojson_name(base, filename, field, output_name):
    path = project_root / 'data' / 'possible_additional_features' / filename
    with path.open() as handle:
        geojson = json.load(handle)
    points = pd.DataFrame([
        {
            'longitude': item['geometry']['coordinates'][0],
            'latitude': item['geometry']['coordinates'][1],
            output_name: item['properties'][field],
        }
        for item in geojson['features']
        if item.get('geometry', {}).get('type') == 'Point'
    ]).dropna()
    _, index = cKDTree(
        points[['longitude', 'latitude']].to_numpy() * [110.96, 111.0]
    ).query(base[['LONGITUDE', 'LATITUDE']].to_numpy() * [110.96, 111.0])
    return pd.Series(points[output_name].astype(object).to_numpy()[index], name=output_name)

def make_best_features(raw_history, raw_future, datasets):
    preprocessor = RentalPreprocessor(datasets)
    history_base = preprocessor.fit_transform(
        raw_history.drop(columns=['period', TARGET], errors='ignore')
    )
    future_base = preprocessor.transform(
        raw_future.drop(columns=['period', TARGET], errors='ignore')
    )
    history = history_base[['TOWN', 'FLAT_TYPE']].reset_index(drop=True).assign(
        period=raw_history['period'].to_numpy(),
        **{TARGET: raw_history[TARGET].to_numpy()},
    )
    future = future_base[['TOWN', 'FLAT_TYPE']].reset_index(drop=True).assign(
        period=raw_future['period'].to_numpy(),
    )
    target_transformer = RentalTargetFeatureTransformer()
    history_target = target_transformer.fit_transform_train(history)
    future_target = target_transformer.transform(future)

    def combine(base, target_features):
        result = pd.concat([
            base[COMPACT_FEATURES].reset_index(drop=True),
            target_features.reset_index(drop=True),
            nearest_mrt_name(base, datasets).reset_index(drop=True),
            nearest_geojson_name(
                base, 'TouristAttractions.geojson', 'PAGETITLE',
                'nearest_tourist_attraction_name',
            ).reset_index(drop=True),
            nearest_geojson_name(
                base, 'MonumentsGEOJSON.geojson', 'NAME',
                'nearest_monument_name',
            ).reset_index(drop=True),
        ], axis=1)
        return result[BEST_FEATURES]

    return combine(history_base, history_target), combine(future_base, future_target)

## Nested chronological HPO and stacking helpers

`fit_hpo_predictor` reserves the latest part of the supplied history as an inner HPO window and refits the selected model on all supplied rows. `chronological_oof_hpo` then predicts later months from earlier months only. The same mechanism is used for the base and both residual targets.

In [11]:
def fit_hpo_predictor(x, y, periods, label, path, trials_per_model, time_limit):
    x = x.reset_index(drop=True)
    y = pd.Series(y).astype(float).reset_index(drop=True)
    periods = pd.Series(periods).reset_index(drop=True)
    unique_periods = np.sort(periods.unique())
    if len(unique_periods) < 3:
        raise ValueError(
            f'Need at least 3 months for chronological HPO; found {len(unique_periods)}.'
        )
    # Later residual layers contain fewer OOF months. Keep at least two months
    # for fitting and use an adaptive final-month tuning window. A single
    # tuning month still contains many listing rows in this dataset.
    tune_months = min(12, max(1, len(unique_periods) // 5))
    tune_periods = unique_periods[-tune_months:]
    tune_mask = periods.isin(tune_periods).to_numpy()
    train_data = x.loc[~tune_mask].assign(**{label: y.loc[~tune_mask].to_numpy()})
    tuning_data = x.loc[tune_mask].assign(**{label: y.loc[tune_mask].to_numpy()})
    predictor = TabularPredictor(
        label=label, problem_type='regression', eval_metric='root_mean_squared_error',
        path=str(path),
    ).fit(
        train_data=train_data,
        tuning_data=tuning_data,
        hyperparameters=HPO_HYPERPARAMETERS,
        hyperparameter_tune_kwargs=hpo_kwargs(trials_per_model),
        presets=['medium_quality', 'ignore_text'],
        time_limit=time_limit,
        verbosity=2,
    )
    predictor.refit_full(model='best')
    return predictor

def chronological_oof_hpo(x, y, periods, label, stage_name):
    x = x.reset_index(drop=True)
    y = pd.Series(y).astype(float).reset_index(drop=True)
    periods = pd.Series(periods).reset_index(drop=True)
    unique_periods = np.sort(periods.unique())
    splitter = TimeSeriesSplit(n_splits=N_SPLITS)
    parts = []
    for fold, (early_period_index, later_period_index) in enumerate(splitter.split(unique_periods), 1):
        early_periods = unique_periods[early_period_index]
        later_periods = unique_periods[later_period_index]
        early_mask = periods.isin(early_periods).to_numpy()
        later_mask = periods.isin(later_periods).to_numpy()
        predictor = fit_hpo_predictor(
            x.loc[early_mask], y.loc[early_mask], periods.loc[early_mask], label,
            MODEL_ROOT / stage_name / f'oof_{fold}',
            OOF_TRIALS_PER_MODEL, OOF_TIME_LIMIT,
        )
        later_index = np.flatnonzero(later_mask)
        prediction = predictor.predict(x.loc[later_mask]).astype(float).to_numpy()
        parts.append(pd.DataFrame({
            'row_index': later_index,
            'period': periods.loc[later_mask].to_numpy(),
            'target': y.loc[later_mask].to_numpy(),
            'prediction': prediction,
            'fold': fold,
        }))
        print(
            f'{stage_name} fold {fold}: fit through {early_periods[-1]}, '
            f'predicted {later_periods[0]} to {later_periods[-1]} ({later_mask.sum():,} rows)'
        )
    return pd.concat(parts, ignore_index=True).sort_values(['period', 'row_index']).reset_index(drop=True)

def calibrated_weight(oof_frame, upper=1.5):
    unique_periods = np.sort(oof_frame['period'].unique())
    calibration_periods = unique_periods[:max(1, len(unique_periods) // 2)]
    calibration = oof_frame.loc[oof_frame['period'].isin(calibration_periods)]
    prediction = calibration['prediction'].to_numpy()
    denominator = float(np.dot(prediction, prediction))
    raw_weight = 0.0 if denominator == 0 else float(
        np.dot(calibration['target'].to_numpy(), prediction) / denominator
    )
    return float(np.clip(raw_weight, 0.0, upper)), raw_weight, calibration_periods

def metric_row(name, actual, prediction):
    return {
        'model': name,
        'RMSE': mean_squared_error(actual, prediction) ** 0.5,
        'MAE': mean_absolute_error(actual, prediction),
        'R2': r2_score(actual, prediction),
    }

## Load data and create the untouched outer holdout

In [5]:
_, datasets = load_datasets(project_root / 'data')
raw = datasets['train'].copy()
raw['period'] = pd.to_datetime(raw['RENT_APPROVAL_DATE'] + '-01').dt.to_period('M')
all_periods = np.sort(raw['period'].unique())
outer_cutoff = all_periods[int(len(all_periods) * 0.80)]
raw_fit = raw.loc[raw['period'] < outer_cutoff].copy()
raw_valid = raw.loc[raw['period'] >= outer_cutoff].copy()
assert raw_fit['period'].max() < raw_valid['period'].min()
print(
    f'Outer cutoff={outer_cutoff}; fit={len(raw_fit):,} rows through {raw_fit.period.max()}; '
    f'validation={len(raw_valid):,} rows from {raw_valid.period.min()} to {raw_valid.period.max()}.'
)
print(f'Active features ({len(BEST_FEATURES)}): {BEST_FEATURES}')

Outer cutoff=2024-05; fit=117,988 rows through 2024-04; validation=32,012 rows from 2024-05 to 2025-03.
Active features (18): ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', 'FLOOR_AREA_SQM', 'approval_date_num', 'approval_month', 'coe_quota_premium_mean', 'YEAR_COMPLETED', 'MAX_FLOOR', 'mrt_dist_nearest_km', 'mall_dist_nearest_km', 'monthly_target_lag', 'town_flat_type_target_mean', 'nearest_mrt_name', 'nearest_tourist_attraction_name', 'nearest_monument_name']


## Layer 1: HPO base ensemble and chronological base OOF predictions

In [6]:
fit_periods = np.sort(raw_fit['period'].unique())
base_parts = []
for fold, (early_period_index, later_period_index) in enumerate(
    TimeSeriesSplit(n_splits=N_SPLITS).split(fit_periods), 1
):
    early_periods = fit_periods[early_period_index]
    later_periods = fit_periods[later_period_index]
    early = raw_fit.loc[raw_fit['period'].isin(early_periods)].copy()
    later = raw_fit.loc[raw_fit['period'].isin(later_periods)].copy()
    early_x, later_x = make_best_features(early, later, datasets)
    predictor = fit_hpo_predictor(
        early_x, early[TARGET], early['period'], TARGET,
        MODEL_ROOT / 'base' / f'oof_{fold}',
        OOF_TRIALS_PER_MODEL, OOF_TIME_LIMIT,
    )
    part = later_x.copy()
    part['period'] = later['period'].to_numpy()
    part['actual'] = later[TARGET].astype(float).to_numpy()
    part['base_prediction'] = predictor.predict(later_x).astype(float).clip(lower=0).to_numpy()
    part['base_fold'] = fold
    base_parts.append(part)
    print(
        f'Base fold {fold}: fit through {early_periods[-1]}, predicted '
        f'{later_periods[0]} to {later_periods[-1]} ({len(later):,} rows)'
    )
base_oof = pd.concat(base_parts, ignore_index=True).sort_values('period').reset_index(drop=True)
base_oof['residual_1_target'] = base_oof['actual'] - base_oof['base_prediction']
print(f'Base OOF rows: {len(base_oof):,}; RMSE={mean_squared_error(base_oof.actual, base_oof.base_prediction) ** 0.5:.3f}')
display(base_oof[['period', 'actual', 'base_prediction', 'residual_1_target']].describe(include='all'))

[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Verbosity: 2 (Standard Logging)


[categorical_clean] derived HDB feature | REGION: 5 -> 5 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.19 GB / 16.00 GB (26.2%)
Disk Space Avail:   172.39 GB / 460.38 GB (37.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 900s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_17_two_residual_hpt/20261020_123932/base/oof_1"
Train Data Rows:    22421
Train Data Columns: 18
Tuning Data Rows:    9144
Tuning Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preprocessing data...
Using F

Base fold 1: fit through 2021-10, predicted 2021-11 to 2022-08 (28,514 rows)
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, col

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.04 GB / 16.00 GB (25.3%)
Disk Space Avail:   172.27 GB / 460.38 GB (37.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 900s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_17_two_residual_hpt/20261020_123932/base/oof_2"
Train Data Rows:    49389
Train Data Columns: 18
Tuning Data Rows:    10690
Tuning Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regress

Base fold 2: fit through 2022-08, predicted 2022-09 to 2023-06 (27,858 rows)
[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, col

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.06 GB / 16.00 GB (25.4%)
Disk Space Avail:   172.13 GB / 460.38 GB (37.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 900s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_17_two_residual_hpt/20261020_123932/base/oof_3"
Train Data Rows:    70183
Train Data Columns: 18
Tuning Data Rows:    17754
Tuning Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regress

Base fold 3: fit through 2023-06, predicted 2023-07 to 2024-04 (30,051 rows)
Base OOF rows: 86,423; RMSE=619.071


,period,actual,base_prediction,residual_1_target
count,86423,86423.000000,86423.000000,86423.000000
unique,30,NaN,NaN,NaN
top,2022-01,NaN,NaN,NaN
freq,3278,NaN,NaN,NaN
mean,NaN,2791.240549,2578.917515,212.323034
std,NaN,743.320096,533.399890,581.524935
min,NaN,300.000000,1150.351807,-3936.496582
25%,NaN,2200.000000,2159.628906,-140.819824
50%,NaN,2800.000000,2506.989990,192.915283
75%,NaN,3300.000000,2933.670288,584.072632


## Residual layer 1

The first residual model is tuned on earlier base-model OOF errors. Its shrinkage weight is calibrated using only the earlier half of its own chronological OOF predictions.

In [7]:
residual_1_oof = chronological_oof_hpo(
    base_oof[BEST_FEATURES],
    base_oof['residual_1_target'],
    base_oof['period'],
    'residual_1_target',
    'residual_1',
)
weight_1, raw_weight_1, calibration_periods_1 = calibrated_weight(residual_1_oof)
print(
    f'Residual layer 1 weight={weight_1:.6f} (raw={raw_weight_1:.6f}); '
    f'calibration through {calibration_periods_1[-1]}.'
)
display(residual_1_oof.groupby('fold').agg(
    rows=('target', 'size'),
    target_mean=('target', 'mean'),
    prediction_mean=('prediction', 'mean'),
))

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.02 GB / 16.00 GB (25.1%)
Disk Space Avail:   171.96 GB / 460.38 GB (37.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 900s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_17_two_residual_hpt/20261020_123932/residual_1/oof_1"
Train Data Rows:    17824
Train Data Columns: 18
Tuning Data Rows:    8274
Tuning Data Columns: 18
Label Column:       residual_1_target
Problem Type:    

residual_1 fold 1: fit through 2022-07, predicted 2022-08 to 2023-02 (18,086 rows)


	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', [])        : 8 | ['FLOOR_AREA_SQM', 'coe_quota_premium_mean', 'YEAR_COMPLETED', 'MAX_FLOOR', 'mrt_dist_nearest_km', ...]
		('int', [])          : 2 | ['approval_date_num', 'approval_month']
		('object', [])       : 7 | ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', ...]
		('object', ['text']) : 1 | ['nearest_mrt_name']
	Types of features in processed data (raw dtype, special dtypes):
		('category', [])                   : 7 | ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', ...]
		('category', ['text_as_category']) : 1 | ['nearest_mrt_name']
		('float', [])                      : 8 | ['FLOOR_AREA_SQM', 'coe_quota_premium_mean

residual_1 fold 2: fit through 2023-02, predicted 2023-03 to 2023-09 (21,116 rows)


	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', [])        : 8 | ['FLOOR_AREA_SQM', 'coe_quota_premium_mean', 'YEAR_COMPLETED', 'MAX_FLOOR', 'mrt_dist_nearest_km', ...]
		('int', [])          : 2 | ['approval_date_num', 'approval_month']
		('object', [])       : 7 | ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', ...]
		('object', ['text']) : 1 | ['nearest_mrt_name']
	Types of features in processed data (raw dtype, special dtypes):
		('category', [])                   : 7 | ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', ...]
		('category', ['text_as_category']

residual_1 fold 3: fit through 2023-09, predicted 2023-10 to 2024-04 (21,123 rows)
Residual layer 1 weight=0.956195 (raw=0.956195); calibration through 2023-05.


,rows,target_mean,prediction_mean
fold,,,
1,18086,321.528904,391.683266
2,21116,321.851981,433.685471
3,21123,30.157078,273.096356


## Residual layer 2

Layer 2 predicts what remains after the OOF base prediction and the shrunken Layer-1 residual correction. Only rows with a genuine Layer-1 OOF prediction are eligible.

In [12]:
residual_2_rows = base_oof.iloc[residual_1_oof['row_index'].to_numpy()].reset_index(drop=True)
residual_2_rows['residual_1_oof_prediction'] = residual_1_oof['prediction'].to_numpy()
residual_2_rows['residual_2_target'] = (
    residual_1_oof['target'].to_numpy()
    - weight_1 * residual_1_oof['prediction'].to_numpy()
)
residual_2_oof = chronological_oof_hpo(
    residual_2_rows[BEST_FEATURES],
    residual_2_rows['residual_2_target'],
    residual_2_rows['period'],
    'residual_2_target',
    'residual_2',
)
weight_2, raw_weight_2, calibration_periods_2 = calibrated_weight(residual_2_oof)
print(
    f'Residual layer 2 weight={weight_2:.6f} (raw={raw_weight_2:.6f}); '
    f'calibration through {calibration_periods_2[-1]}.'
)
display(residual_2_oof.groupby('fold').agg(
    rows=('target', 'size'),
    target_mean=('target', 'mean'),
    prediction_mean=('prediction', 'mean'),
))

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.98 GB / 16.00 GB (24.9%)
Disk Space Avail:   171.76 GB / 460.38 GB (37.3%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 900s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_17_two_residual_hpt/20261020_123932/residual_2/oof_1"
Train Data Rows:    12520
Train Data Columns: 18
Tuning Data Rows:    2622
Tuning Data Columns: 18
Label Column:       residual_2_target
Problem Type:    

residual_2 fold 1: fit through 2023-01, predicted 2023-02 to 2023-06 (15,132 rows)


Data preprocessing and feature engineering runtime = 0.13s ...
AutoGluon will gauge predictive performance using evaluation metric: 'root_mean_squared_error'
	This metric's sign has been flipped to adhere to being higher_is_better. The metric score can be multiplied by -1 to get the metric value.
	To change this, specify the eval_metric parameter of Predictor()
User-specified model hyperparameters to be fit:
{
	'GBM': [{'num_leaves': Int: lower=24, upper=500, 'num_iterations': Int: lower=100, upper=1000, 'learning_rate': Real: lower=0.01, upper=0.12, 'min_data_in_leaf': Int: lower=20, upper=250, 'feature_fraction': Real: lower=0.7, upper=1.0, 'bagging_fraction': Real: lower=0.7, upper=1.0, 'bagging_freq': 1, 'lambda_l1': Real: lower=0.0001, upper=20.0, 'lambda_l2': Real: lower=0.0001, upper=20.0}],
	'CAT': [{'depth': Int: lower=5, upper=10, 'iterations': Int: lower=100, upper=1000, 'learning_rate': Real: lower=0.01, upper=0.12, 'l2_leaf_reg': Real: lower=1.0, upper=30.0, 'random_streng

residual_2 fold 2: fit through 2023-06, predicted 2023-07 to 2023-11 (14,892 rows)


			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', [])        : 8 | ['FLOOR_AREA_SQM', 'coe_quota_premium_mean', 'YEAR_COMPLETED', 'MAX_FLOOR', 'mrt_dist_nearest_km', ...]
		('int', [])          : 2 | ['approval_date_num', 'approval_month']
		('object', [])       : 7 | ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', ...]
		('object', ['text']) : 1 | ['nearest_mrt_name']
	Types of features in processed data (raw dtype, special dtypes):
		('category', [])                   : 7 | ['TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL', ...]
		('category', ['text_as_category']) : 1 | ['nearest_mrt_name']
		('float', [])                      : 8 | ['FLOOR_AREA_SQM', 'coe_quota_premium_mean', 'YEAR_COMPLETED', 'MAX_FLOOR', 'mrt_dist_nearest_km', ...]
		('int', [])                      

residual_2 fold 3: fit through 2023-11, predicted 2023-12 to 2024-04 (15,159 rows)
Residual layer 2 weight=0.000000 (raw=-0.440466); calibration through 2023-08.


,rows,target_mean,prediction_mean
fold,,,
1,15132,121.728153,54.372135
2,14892,-317.904617,30.863673
3,15159,-227.859564,-166.590805


## Final HPO fits and untouched validation comparison

The three final predictors are independently tuned and refitted. Residual models learn only from OOF errors; the untouched validation labels are used once, below, for reporting.

In [13]:
fit_x, valid_x = make_best_features(raw_fit, raw_valid, datasets)
base_model = fit_hpo_predictor(
    fit_x, raw_fit[TARGET], raw_fit['period'], TARGET,
    MODEL_ROOT / 'base' / 'final',
    FINAL_TRIALS_PER_MODEL, FINAL_TIME_LIMIT,
)
residual_1_model = fit_hpo_predictor(
    base_oof[BEST_FEATURES], base_oof['residual_1_target'], base_oof['period'],
    'residual_1_target', MODEL_ROOT / 'residual_1' / 'final',
    FINAL_TRIALS_PER_MODEL, FINAL_TIME_LIMIT,
)
residual_2_model = fit_hpo_predictor(
    residual_2_rows[BEST_FEATURES], residual_2_rows['residual_2_target'],
    residual_2_rows['period'], 'residual_2_target',
    MODEL_ROOT / 'residual_2' / 'final',
    FINAL_TRIALS_PER_MODEL, FINAL_TIME_LIMIT,
)

actual = raw_valid[TARGET].astype(float).to_numpy()
base_prediction = base_model.predict(valid_x).astype(float).clip(lower=0).to_numpy()
residual_1_prediction = residual_1_model.predict(valid_x).astype(float).to_numpy()
residual_2_prediction = residual_2_model.predict(valid_x).astype(float).to_numpy()
stage_1_prediction = np.clip(base_prediction + weight_1 * residual_1_prediction, 0, None)
stage_2_prediction = np.clip(
    base_prediction
    + weight_1 * residual_1_prediction
    + weight_2 * residual_2_prediction,
    0, None,
)
comparison = pd.DataFrame([
    metric_row('HPO base AutoGluon', actual, base_prediction),
    metric_row(f'+ residual 1 (weight={weight_1:.3f})', actual, stage_1_prediction),
    metric_row(f'+ residual 2 (weight={weight_2:.3f})', actual, stage_2_prediction),
]).sort_values('RMSE').reset_index(drop=True)
display(comparison)
print(f'Best validation stage: {comparison.iloc[0].model}; RMSE={comparison.iloc[0].RMSE:.6f}')

[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       4.13 GB / 16.00 GB (25.8%)
Disk Space Avail:   171.60 GB / 460.38 GB (37.3%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 1800s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_17_two_residual_hpt/20261020_123932/base/final"
Train Data Rows:    94118
Train Data Columns: 18
Tuning Data Rows:    23870
Tuning Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regres

,model,RMSE,MAE,R2
0,HPO base AutoGluon,470.231716,343.270955,0.487725
1,+ residual 1 (weight=0.956),474.426051,341.801835,0.478545
2,+ residual 2 (weight=0.000),474.426051,341.801835,0.478545


Best validation stage: HPO base AutoGluon; RMSE=470.231716


## Diagnostics and saved validation artifacts

The leaderboard for each final layer shows which HPO trials and ensemble won. Validation predictions and metrics are saved so that the exact Exp. 17 result remains available for the report.

In [14]:
for name, predictor in {
    'base': base_model,
    'residual_1': residual_1_model,
    'residual_2': residual_2_model,
}.items():
    print(f'\n{name} leaderboard')
    display(predictor.leaderboard(silent=True).head(15))

validation_predictions = pd.DataFrame({
    'row_index': raw_valid.index,
    'period': raw_valid['period'].astype(str).to_numpy(),
    'actual': actual,
    'base_prediction': base_prediction,
    'residual_1_prediction': residual_1_prediction,
    'residual_1_weight': weight_1,
    'residual_2_prediction': residual_2_prediction,
    'residual_2_weight': weight_2,
    'final_prediction': stage_2_prediction,
})
prediction_path = experiment_dir / 'exp_17_two_residual_hpt_validation_predictions.csv'
metrics_path = experiment_dir / 'exp_17_two_residual_hpt_metrics.csv'
validation_predictions.to_csv(prediction_path, index=False)
comparison.to_csv(metrics_path, index=False)
print(f'Saved validation predictions to {prediction_path}')
print(f'Saved metrics to {metrics_path}')


base leaderboard


,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-545.694574,root_mean_squared_error,0.505401,143.838038,0.000323,0.019623,2,True,61
1,CatBoost/T8,-546.830310,root_mean_squared_error,0.038583,37.252125,0.038583,37.252125,1,True,38
2,CatBoost/T30,-546.922640,root_mean_squared_error,0.031512,13.993539,0.031512,13.993539,1,True,60
3,CatBoost/T17,-546.953843,root_mean_squared_error,0.032575,16.590916,0.032575,16.590916,1,True,47
4,CatBoost/T25,-547.001147,root_mean_squared_error,0.044638,30.755488,0.044638,30.755488,1,True,55
5,CatBoost/T19,-547.047734,root_mean_squared_error,0.031056,18.016730,0.031056,18.016730,1,True,49
6,CatBoost/T13,-547.236356,root_mean_squared_error,0.034004,22.362415,0.034004,22.362415,1,True,43
7,CatBoost/T20,-547.755500,root_mean_squared_error,0.033873,13.852401,0.033873,13.852401,1,True,50
8,CatBoost/T3,-548.034237,root_mean_squared_error,0.031175,17.567194,0.031175,17.567194,1,True,33
9,CatBoost/T2,-548.831542,root_mean_squared_error,0.028713,15.516624,0.028713,15.516624,1,True,32



residual_1 leaderboard


,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-547.190075,root_mean_squared_error,0.114931,17.110331,0.000225,0.012334,2,True,61
1,CatBoost/T10,-549.756057,root_mean_squared_error,0.014700,1.026058,0.014700,1.026058,1,True,40
2,CatBoost/T7,-550.890887,root_mean_squared_error,0.023934,8.503511,0.023934,8.503511,1,True,37
3,CatBoost/T15,-551.183891,root_mean_squared_error,0.021806,11.718625,0.021806,11.718625,1,True,45
4,CatBoost/T13,-551.378988,root_mean_squared_error,0.027143,12.311048,0.027143,12.311048,1,True,43
5,CatBoost/T9,-551.697455,root_mean_squared_error,0.018536,5.396468,0.018536,5.396468,1,True,39
6,LightGBM/T18,-551.774986,root_mean_squared_error,0.015492,4.001491,0.015492,4.001491,1,True,18
7,LightGBM/T5,-551.817534,root_mean_squared_error,0.076072,7.568428,0.076072,7.568428,1,True,5
8,LightGBM/T23,-552.269013,root_mean_squared_error,0.008338,2.062579,0.008338,2.062579,1,True,23
9,CatBoost/T4,-552.271403,root_mean_squared_error,0.019540,11.558334,0.019540,11.558334,1,True,34



residual_2 leaderboard


,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-534.591665,root_mean_squared_error,0.045662,23.573821,0.000152,0.007368,2,True,61
1,LightGBM/T8,-536.222275,root_mean_squared_error,0.005770,2.383398,0.005770,2.383398,1,True,8
2,LightGBM/T15,-536.320481,root_mean_squared_error,0.008774,3.133070,0.008774,3.133070,1,True,15
3,CatBoost/T18,-536.349994,root_mean_squared_error,0.026507,15.390705,0.026507,15.390705,1,True,48
4,LightGBM/T27,-536.379269,root_mean_squared_error,0.008187,2.310156,0.008187,2.310156,1,True,27
5,LightGBM/T18,-536.406019,root_mean_squared_error,0.004459,2.659280,0.004459,2.659280,1,True,18
6,LightGBM/T29,-536.498609,root_mean_squared_error,0.003974,1.717057,0.003974,1.717057,1,True,29
7,LightGBM/T23,-536.586198,root_mean_squared_error,0.003201,1.491458,0.003201,1.491458,1,True,23
8,LightGBM/T20,-536.622523,root_mean_squared_error,0.004442,1.907518,0.004442,1.907518,1,True,20
9,LightGBM/T9,-536.650042,root_mean_squared_error,0.016597,2.311385,0.016597,2.311385,1,True,9


Saved validation predictions to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_17_two_residual_hpt_validation_predictions.csv
Saved metrics to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_17_two_residual_hpt_metrics.csv


## Runtime guidance

The defaults are intended for a serious final run: 12 trials per model family in each OOF fit and 30 per family in each final layer. With three folds and three layers, this can take several hours. For a smoke test, set `EXP17_OOF_TRIALS_PER_MODEL=2`, `EXP17_FINAL_TRIALS_PER_MODEL=3`, `EXP17_OOF_TIME_LIMIT=120`, and `EXP17_FINAL_TIME_LIMIT=300` before starting the kernel. The smoke-test score must not be reported as the final Exp. 17 result.